# Machine Learning Models for Severity Classification

## Logistic regreassion

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
df = pd.read_csv("../data/clean_airline_reviews.csv")

df.head()

In [ ]:
df["Overall_Rating"] = pd.to_numeric(df["Overall_Rating"], errors="coerce")

In [ ]:
df = df.dropna(subset=["Overall_Rating"])

df.shape

In [ ]:
def severity_label(rating):
    if rating <= 2:
        return "Critical"
    elif rating <= 4:
        return "High"
    elif rating <= 6:
        return "Medium"
    else:
        return "Low"

In [ ]:
df["severity_label"] = df["Overall_Rating"].apply(severity_label)

df["severity_label"].value_counts()

In [ ]:
label_encoder = LabelEncoder()

df["severity_id"] = label_encoder.fit_transform(df["severity_label"])

df[["severity_label", "severity_id"]].head()

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1,2)
)

In [ ]:
X = vectorizer.fit_transform(df["final_text"])
y = df["severity_id"]

X.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

In [ ]:
for i, label in enumerate(label_encoder.classes_):
    print(f"{i}: {label}")

## Naive Bayes

In [ ]:
from sklearn.naive_bayes import MultinomialNB

In [ ]:
nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred_nb))

In [ ]:
from sklearn.svm import LinearSVC

In [ ]:
svm_model = LinearSVC()

svm_model.fit(X_train, y_train)

In [ ]:
y_pred_svm = svm_model.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred_svm))

## Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

def plot_confusion_matrix(y_test, y_pred, model_name):
    
    cm = confusion_matrix(y_test, y_pred)
    
    plt.figure(figsize=(6,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    
    plt.title(f"{model_name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    
    plt.show()

In [ ]:
plot_confusion_matrix(y_test, y_pred, "Logistic Regression")

In [ ]:
plot_confusion_matrix(y_test, y_pred_nb, "Naive Bayes")

In [ ]:
plot_confusion_matrix(y_test, y_pred_svm, "SVM")

## Model Comparison

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

In [ ]:
def get_metrics(y_test, y_pred):
    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average='macro'),
        "Recall": recall_score(y_test, y_pred, average='macro'),
        "F1 Score": f1_score(y_test, y_pred, average='macro')
    }

In [ ]:
results = []

# Logistic Regression
results.append({
    "Model": "Logistic Regression",
    **get_metrics(y_test, y_pred)
})

# Naive Bayes
results.append({
    "Model": "Naive Bayes",
    **get_metrics(y_test, y_pred_nb)
})

# SVM
results.append({
    "Model": "SVM",
    **get_metrics(y_test, y_pred_svm)
})

results_df = pd.DataFrame(results)

In [ ]:
results_df


## Model Comparison Conclusion

Among the three models tested (Logistic Regression, Naive Bayes, and SVM), SVM performed the best overall.

Although Logistic Regression and Naive Bayes achieved slightly higher accuracy, they showed poor performance in minority classes due to dataset imbalance.

SVM achieved the highest F1-score and better recall across classes, making it the most balanced and reliable model for severity classification.

Therefore, SVM is selected as the best performing traditional machine learning model for this task.